In [ ]:

import zipfile
import os
import cv2

zip_path = "/content/nlp-getting-started.zip"
extract_path = "disaster_tweet"
# Extract the ZIP file
with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_path)

In [ ]:
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

# Load the dataset
# Make sure 'train.csv' is uploaded to your Colab session
df = pd.read_csv('/content/disaster_tweet/train.csv')

# Take a peek at the first 5 rows
df.head()

,id,keyword,location,text,target
0,1,NaN,NaN,Our Deeds are the Reason of this #earthquake M...,1
1,4,NaN,NaN,Forest fire near La Ronge Sask. Canada,1
2,5,NaN,NaN,All residents asked to 'shelter in place' are ...,1
3,6,NaN,NaN,"13,000 people receive #wildfires evacuation or...",1
4,7,NaN,NaN,Just got sent this photo from Ruby #Alaska as ...,1


In [ ]:
def clean_tweet(text):
    # Remove URLs
    text = re.sub(r'http\S+', '', text)
    # Remove mentions (@username)
    text = re.sub(r'@\w+', '', text)
    # Remove hashtags (just the # symbol, keep the word)
    text = re.sub(r'#', '', text)
    # Remove special characters and numbers
    text = re.sub(r'[^A-Za-z\s]', '', text)
    # Convert to lowercase
    text = text.lower()
    return text

# Apply the cleaning function to the 'text' column
df['cleaned_text'] = df['text'].apply(clean_tweet)

# Check out the cleaned versions
print(df[['text', 'cleaned_text']].head())

                                                text  \
0  Our Deeds are the Reason of this #earthquake M...   
1             Forest fire near La Ronge Sask. Canada   
2  All residents asked to 'shelter in place' are ...   
3  13,000 people receive #wildfires evacuation or...   
4  Just got sent this photo from Ruby #Alaska as ...   

                                        cleaned_text  
0  our deeds are the reason of this earthquake ma...  
1              forest fire near la ronge sask canada  
2  all residents asked to shelter in place are be...  
3   people receive wildfires evacuation orders in...  
4  just got sent this photo from ruby alaska as s...  


In [ ]:
# Split the data into features (X) and target labels (y)
X = df['cleaned_text']
y = df['target']

# Split into training data (80%) and validation data (20%)
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# Initialize the TF-IDF Vectorizer
vectorizer = TfidfVectorizer(max_features=5000, stop_words='english')

# Fit the vectorizer on training data and transform both train and validation sets
X_train_vec = vectorizer.fit_transform(X_train)
X_val_vec = vectorizer.transform(X_val)

print(f"Training data shape: {X_train_vec.shape}")

Training data shape: (6090, 5000)


In [ ]:
# Initialize and train the model
model = LogisticRegression(max_iter=1000)
model.fit(X_train_vec, y_train)

print("Model training complete!")

Model training complete!


In [ ]:
# Make predictions on the validation set
y_pred = model.predict(X_val_vec)

# Check the accuracy
accuracy = accuracy_score(y_val, y_pred)
print(f"Model Accuracy: {accuracy * 100:.2f}%\n")

# Print a detailed classification report
print("Classification Report:")
print(classification_report(y_val, y_pred))

Model Accuracy: 79.97%

Classification Report:
              precision    recall  f1-score   support

           0       0.79      0.89      0.84       874
           1       0.82      0.68      0.74       649

    accuracy                           0.80      1523
   macro avg       0.80      0.78      0.79      1523
weighted avg       0.80      0.80      0.80      1523



In [ ]:
def predict_tweet(custom_tweet):
    cleaned = clean_tweet(custom_tweet)
    vectorized = vectorizer.transform([cleaned])
    prediction = model.predict(vectorized)

    if prediction[0] == 1:
        return "🚨 Real Disaster Tweet"
    else:
        return "✅ Not a Disaster"

# Try it out!
print(predict_tweet("There is a massive forest fire heading toward the city!"))
print(predict_tweet("This new action movie is absolute fire, I loved it!"))

🚨 Real Disaster Tweet
✅ Not a Disaster


In [ ]:
# Create a simple interactive loop
print("--- Disaster Tweet Predictor ---")
print("Type 'exit' to stop.\n")

while True:
    user_input = input("Enter a tweet to classify: ")

    # Check if the user wants to quit
    if user_input.lower() == 'exit':
        print("Stopping the predictor.")
        break

    # Get the prediction
    result = predict_tweet(user_input)

    # Display the result
    print(f"➔ {result}\n")

--- Disaster Tweet Predictor ---
Type 'exit' to stop.

Enter a tweet to classify: this movie is a disaster
➔ 🚨 Real Disaster Tweet

Enter a tweet to classify: flood came all over kerala
➔ 🚨 Real Disaster Tweet

Enter a tweet to classify: hoi
➔ ✅ Not a Disaster

Enter a tweet to classify: exit
Stopping the predictor.


In [ ]:
import ipywidgets as widgets
from IPython.display import display

# 1. Create the text input box
tweet_input = widgets.Text(
    value='',
    placeholder='Type a tweet here...',
    description='Tweet:',
    layout=widgets.Layout(width='80%')
)

# 2. Create the predict button
predict_button = widgets.Button(
    description='Predict',
    button_style='info', # Gives the button a nice blue color
    tooltip='Click to classify the tweet'
)

# 3. Create an output area to show the result
output_area = widgets.Output()

# 4. Define what happens when the button is clicked
def on_button_click(b):
    with output_area:
        output_area.clear_output() # Clear previous results
        text = tweet_input.value
        if text.strip() == "":
            print("Please enter some text!")
        else:
            result = predict_tweet(text)
            print(f"Analyzing: '{text}'")
            print(f"Result: {result}")

# Attach the function to the button
predict_button.on_click(on_button_click)

# 5. Display the UI components on the screen
print("Interactive Disaster Tweet Classifier 🚨")
display(tweet_input, predict_button, output_area)

Interactive Disaster Tweet Classifier 🚨


Text(value='', description='Tweet:', layout=Layout(width='80%'), placeholder='Type a tweet here...')

Button(button_style='info', description='Predict', style=ButtonStyle(), tooltip='Click to classify the tweet')

Output()